# 03 — DL Experiments: CNN-LSTM & CNN-BiLSTM-Attention

Chay thi nghiem DL voi 2 kien truc:
- `cnn_lstm`: CNN-LSTM don gian (tuong tu Villa 2025)
- `cnn_bilstm_attention`: CNN + Bidirectional LSTM + Multi-Head Attention (tuong tu JMIR 2024)

**Yeu cau:** Can cai dat TensorFlow >= 2.10
```
pip install tensorflow
```


In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from config import SENSOR_VARIANTS, TRAIN_SUBJECTS, VAL_SUBJECTS, TEST_SA, TEST_SE
from windowing import build_window_dataset, subject_split
from dl_models import DLTrainer, run_dl_models
from ml_models import compute_metrics, print_metrics
from evaluate import (plot_confusion_matrix, plot_roc_curve,
                       plot_sensor_variant_comparison, make_results_table)

import tensorflow as tf
print(f'TF version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## Buoc 1: Doc du lieu (ALL9 truoc)

In [ ]:
all_subjects = TRAIN_SUBJECTS + VAL_SUBJECTS + TEST_SA + TEST_SE
sensor_cols  = SENSOR_VARIANTS['ALL9']

X, y, meta = build_window_dataset(
    subjects=all_subjects,
    sensor_cols=sensor_cols,
    verbose=True)

splits = subject_split(X, y, meta)

X_train = splits['train']['X'].astype(np.float32)
y_train = splits['train']['y']
X_val   = splits['val']['X'].astype(np.float32)
y_val   = splits['val']['y']
X_test  = splits['test']['X'].astype(np.float32)
y_test  = splits['test']['y']

print(f'Input shape: {X_train.shape}  (N, window_size, channels)')

## Buoc 2: Train CNN-LSTM (baseline DL)

In [ ]:
trainer_cnn_lstm = DLTrainer(arch='cnn_lstm')
trainer_cnn_lstm.fit(X_train, y_train, X_val, y_val)
trainer_cnn_lstm.plot_history()

In [ ]:
# Danh gia
test_metrics_cnn = trainer_cnn_lstm.evaluate(X_test, y_test, 'Test')

# Confusion matrix
y_pred, y_prob = trainer_cnn_lstm.predict(X_test)
plot_confusion_matrix(y_test, y_pred, title='CNN-LSTM — ALL9 — Test',
                       save_name='cm_cnn_lstm_ALL9.png')
plot_roc_curve(y_test, y_prob, label='CNN-LSTM ALL9', save_name='roc_cnn_lstm_ALL9.png')

## Buoc 3: Train CNN-BiLSTM + Attention (model nang cao)

In [ ]:
trainer_attn = DLTrainer(arch='cnn_bilstm_attention')
trainer_attn.fit(X_train, y_train, X_val, y_val)
trainer_attn.plot_history()

In [ ]:
test_metrics_attn = trainer_attn.evaluate(X_test, y_test, 'Test')
y_pred_a, y_prob_a = trainer_attn.predict(X_test)
plot_confusion_matrix(y_test, y_pred_a, title='CNN-BiLSTM-Attention — ALL9',
                       save_name='cm_cnn_bilstm_ALL9.png')
plot_roc_curve(y_test, y_prob_a, label='CNN-BiLSTM-Attention', save_name='roc_attn_ALL9.png')

## Buoc 4: DL Sensor Variants (tuy chon — mat nhieu thoi gian)

In [ ]:
# Uncomment de chay toan bo (khoang 2-4 gio neu co GPU)

# dl_results = []
# for variant_name, sensor_cols in SENSOR_VARIANTS.items():
#     print(f'DL variant: {variant_name}')
#     X_v, y_v, meta_v = build_window_dataset(
#         subjects=all_subjects, sensor_cols=sensor_cols, verbose=False)
#     sp = subject_split(X_v, y_v, meta_v)
#     res = run_dl_models(
#         sp['train']['X'], sp['train']['y'],
#         sp['val']['X'],   sp['val']['y'],
#         sp['test']['X'],  sp['test']['y'],
#         experiment_name=f'dl_variant_{variant_name}'
#     )
#     dl_results.append(res)
# dl_df = pd.concat(dl_results)
# dl_df.to_csv('../results/metrics/dl_results.csv', index=False)

print('Uncomment cell tren de chay DL sensor variants.')

## Buoc 5: So sanh ket qua voi bai bao 2024-2025

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Ket qua cua chung ta
our_results = pd.DataFrame([
    {'Source': 'Ours: CNN-LSTM',        'Accuracy': test_metrics_cnn['accuracy']*100,
     'Sensitivity': test_metrics_cnn['sensitivity']*100,
     'Specificity': test_metrics_cnn['specificity']*100,
     'Input': 'ALL9', 'Type': 'Ours'},
    {'Source': 'Ours: CNN-BiLSTM-Attn', 'Accuracy': test_metrics_attn['accuracy']*100,
     'Sensitivity': test_metrics_attn['sensitivity']*100,
     'Specificity': test_metrics_attn['specificity']*100,
     'Input': 'ALL9', 'Type': 'Ours'},
])

# Baseline tu bai bao
papers = pd.DataFrame([
    {'Source': 'Villa 2025 (Sensors)',    'Accuracy': 98.9,  'Sensitivity': 96.7, 'Specificity': 99.6, 'Input': 'ADXL', 'Type': 'Paper'},
    {'Source': 'Kavuncuoglu 2024 (CI)',   'Accuracy': 98.69, 'Sensitivity': 98.28,'Specificity': 99.08,'Input': 'ADXL', 'Type': 'Paper'},
    {'Source': 'Zhang 2024 (JMIR)',       'Accuracy': 99.32, 'Sensitivity': 99.15,'Specificity': None,  'Input': 'Accel+Gyro', 'Type': 'Paper'},
])

combined = pd.concat([papers, our_results], ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
colors = {'Paper': '#95a5a6', 'Ours': '#e74c3c'}

for ax, metric in zip(axes, ['Accuracy', 'Sensitivity', 'Specificity']):
    sub = combined.dropna(subset=[metric])
    bars = ax.barh(sub['Source'], sub[metric],
                   color=[colors[t] for t in sub['Type']],
                   edgecolor='black', height=0.6)
    ax.set_xlim(sub[metric].min() - 1, 101)
    ax.set_title(metric + ' (%)')
    ax.set_xlabel('%')
    for bar, val in zip(bars, sub[metric]):
        ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}%', va='center', fontsize=9)

legend_patches = [
    mpatches.Patch(color='#95a5a6', label='Prior work'),
    mpatches.Patch(color='#e74c3c', label='Our method'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=2, fontsize=11)
plt.suptitle('So sanh voi cac bai bao tren SisFall (2024-2025)', fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig('../results/figures/comparison_with_papers.png', dpi=150, bbox_inches='tight')
plt.show()